# Hyperparameter tuning on Colab


In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

!pip install optuna
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import optuna
import xgboost as xgb
import lightgbm as lgb
import scipy.stats as stats
from sklearn.model_selection import GroupKFold # determine what type of CV split to do
from sklearn.metrics import mean_absolute_error
from scipy.ndimage import binary_erosion
from skimage.morphology import local_maxima
from skimage.measure import label
from PIL import Image
from pathlib import Path
from lightgbm import early_stopping
%matplotlib inline

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 29.7 MB/s eta 0:00:00


Load repo files.

In [4]:
# Load files
from google.colab import files
files.upload()

# Install and configure kaggle API
!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download staix dataset
!kaggle competitions download -c stai-x-challenge-2026

# Make directory
!mkdir -p data
!unzip -q stai-x-challenge-2026.zip -d data

# Download repo
!git clone -b tree-eddy https://github.com/lennardpische/staix26_seasthemoment.git
%cd staix26_seasthemoment

# Add repo scripts to path
sys.path.append("/content/staix26_seasthemoment-src")


KeyboardInterrupt: 

Load all data.

In [3]:
sys.path.append("/content/staix26_seasthemoment-src")

from src.data_loader import (
    find_data_dir,
    load_train_data,
    load_val_data,
    load_train_pngs,
    load_val_pngs
)

# Get data directory
# Update function to be compatible with colab
root = Path("/content/data")

# Get training data and validation covariates
train_all_drugs, train_all_opioids, train_all_stims = load_train_data(root)
val_cov = load_val_data(root)

# Get image data
train_imgs, train_img_names = load_train_pngs(root)
val_imgs, val_img_names = load_val_pngs(root)

Feature engineering pipeline.

In [4]:
from src.features import create_all_features

# Run feature engineering pipeline on train and val
train_all_drugs = create_all_features(
    train_all_drugs,
    train_imgs,
    train_img_names
)
train_all_opioids = create_all_features(
    train_all_opioids,
    train_imgs,
    train_img_names
)
train_all_stims = create_all_features(
    train_all_stims,
    train_imgs,
    train_img_names
)
X_val = create_all_features(
    val_cov,
    val_imgs,
    val_img_names
)

Extract covariates and prediction target.

In [8]:
from src.tuning import return_covs_and_target

X_train_drugs, y_train_drugs = return_covs_and_target(train_all_drugs)
X_train_opioids, y_train_opioids = return_covs_and_target(train_all_opioids)
X_train_stims, y_train_stims = return_covs_and_target(train_all_stims)

Write GPU oriented functions.

In [9]:
def make_xgb_objective(X_train, y_train, num_folds):
    """
    Constructs an objective function using specified data to pass to Optuna study
    """
    # Convert to categorical just in case
    cat_cols = ["period_id", "jurisdiction", "region", "text_presence"]

    for col in cat_cols:
        X_train[col] = X_train[col].astype("category")

    def xgb_objective(trial):
        """
        XGBoost regressor objective function for Optuna. Native Optuna does not support correct early stopping with k-fold CV.
        It requires that we define a fixed validation set used to eavluate early stopping for ALL folds. So, this is a
        workaround that allows each fold to use its corresponding validation set for early stopping.

        Args:
            trial : Parameter for optuna optimize function

        Returns:
            score : Mean MAE across all k folds
        """
        # Define the hyperparameter search space
        params = {
            # Fixed
            "n_estimators": 3000,
            "early_stopping_rounds": 50,
            "random_state": 111,
            "tree_method": "hist",
            "device": "cuda",
            "enable_categorical": True, # native handling

            # Tunable
            "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.1, log = True),
            "subsample": trial.suggest_float("subsample", 0.5, 1, step = 0.1),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1, step = 0.1),
            "max_depth": trial.suggest_int("max_depth", 2, 10, step = 1),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 7, step = 2),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-6, 30, log = True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-6, 30, log = True),

            # Eval (use MAE)
            "objective": "reg:absoluteerror",
            "eval_metric": "mae"
        }

        # Find groups
        groups = X_train["period_id"]
        X_train_new = X_train.drop(columns = "period_id")

        # Initialize k-fold CV splitter
        cv = GroupKFold(
            n_splits = num_folds,
            shuffle = True,
            random_state = 111,
        )

        # Store MAE across folds
        scores = []

        # Manual loop for each fold
        for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X_train_new, y_train, groups = groups)):

            # Load the train/val sets for the current fold
            X_train_curr = X_train_new.iloc[train_idx]
            y_train_curr = y_train.iloc[train_idx]
            X_val_curr = X_train_new.iloc[val_idx]
            y_val_curr = y_train.iloc[val_idx]

            # Initialize XGB classifier model with params
            model = xgb.XGBRegressor(**params)

            # Train the model and predict on current validation set
            model.fit(X_train_curr, y_train_curr, eval_set = [(X_val_curr, y_val_curr)], verbose = False)
            y_pred = model.predict(X_val_curr)

            # Calculate MAE for current fold
            fold_score = mean_absolute_error(y_val_curr, y_pred)
            scores.append(fold_score)

            # Pruner
            intermediate_score = np.mean(scores)
            trial.report(intermediate_score, step = fold_idx)

            if trial.should_prune():
              raise optuna.TrialPruned()

        # Calculate mean MAE across folds
        score = np.mean(scores)

        return score

    return xgb_objective


def make_lgb_objective(X_train, y_train, num_folds):
    """
    Constructs an objective function using specified data to pass to Optuna study
    """
    # Convert to categorical just in case
    cat_cols = ["period_id", "jurisdiction", "region", "text_presence"]

    for col in cat_cols:
        X_train[col] = X_train[col].astype("category")

    def lgb_objective(trial):
        """
        LightGBM regressor objective function for Optuna. Native Optuna does not support correct early stopping with k-fold CV.
        It requires that we define a fixed validation set used to eavluate early stopping for ALL folds. So, this is a
        workaround that allows each fold to use its corresponding validation set for early stopping.

        Args:
            trial : Parameter for optuna optimize function

        Returns:
            score : Mean MAE across all k folds
        """
        # Define the hyperparameter search space
        params = {
            # Fixed
            "n_estimators": 2000,
            "random_state": 111,
            "bagging_freq": 1,
            "device_type": "gpu",
            "verbose": -1

            # Tunable
            "num_leaves": trial.suggest_int("num_leaves", 15, 127, log = True),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 5, 60),
            "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.1, log = True),
            "subsample": trial.suggest_float("subsample", 0.5, 1, step = 0.1),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1, step = 0.1),
            "max_depth": trial.suggest_int("max_depth", 6, 10, step = 1),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 7, step = 2),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-6, 10, log = True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-6, 10, log = True),

            # Eval (use MAE)
            "objective": "regression_l1",
            "metric": "mae"
        }

        # Find groups
        groups = X_train["period_id"]
        X_train_new = X_train.drop(columns = "period_id")

        # Initialize k-fold CV splitter
        cv = GroupKFold(
            n_splits = num_folds,
            shuffle = True,
            random_state = 111,
        )

        # Store MAE across folds
        scores = []

        # Manual loop for each fold
        for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X_train_new, y_train, groups = groups)):

            # Load the train/val sets for the current fold
            X_train_curr = X_train_new.iloc[train_idx]
            y_train_curr = y_train.iloc[train_idx]
            X_val_curr = X_train_new.iloc[val_idx]
            y_val_curr = y_train.iloc[val_idx]

            # Initialize XGB classifier model with params
            model = lgb.LGBMRegressor(**params)

            # Train the model and predict on current val set w/ early stopping
            model.fit(
                X_train_curr,
                y_train_curr,
                eval_set = [(X_val_curr, y_val_curr)],
                callbacks = [early_stopping(stopping_rounds = 50, verbose = False)] # early stopping
                )
            y_pred = model.predict(X_val_curr)

            # Calculate MAE for current fold
            fold_score = mean_absolute_error(y_val_curr, y_pred)
            scores.append(fold_score)

            # Pruner
            intermediate_score = np.mean(scores)
            trial.report(intermediate_score, step = fold_idx)

            if trial.should_prune():
              raise optuna.TrialPruned()

        # Calculate mean MAE across folds
        score = np.mean(scores)

        return score

    return lgb_objective


def find_best_regressor(X_train, y_train, num_folds):
    """
    Function to find and report the best regressor out of LGB, XGB
    for a given training set

    Args:
        X_train : Features
        y_train : Prediction target
        num_folds : # folds for CV during hyperparameter tuning

    Returns:
        best_model : name of best model type
        best_mean_MAE : best score (mean MAE across k folds)
        best_params : best parameters winning model
    """

    # Create median pruner
    median_pruner = optuna.pruners.MedianPruner(
        n_startup_trials = 10, n_warmup_steps = 2, interval_steps = 1
    )

    # Initialize studies
    xgb_study = optuna.create_study(direction = "minimize", pruner = median_pruner)
    lgb_study = optuna.create_study(direction = "minimize", pruner = median_pruner)

    # Initialize objectives
    xgb_objective = make_xgb_objective(X_train, y_train, num_folds)
    lgb_objective = make_lgb_objective(X_train, y_train, num_folds)

    # Run studies
    lgb_study.optimize(lgb_objective, n_trials = 45, n_jobs = 1)
    xgb_study.optimize(xgb_objective, n_trials = 45, n_jobs = 1)

    # Store completed studies
    model_names = ["XGBoost", "LightGBM"]
    completed_studies = [xgb_study, lgb_study]
    mean_scores = [study.best_value for study in completed_studies]

    # Find best MAE and return those params
    best_idx = np.argmin(mean_scores)
    best_model = model_names[best_idx]
    best_mean_MAE = mean_scores[best_idx]
    best_params = completed_studies[best_idx].best_params

    return best_model, best_mean_MAE, best_params

Hyperparameter tuning and model selection.


In [ ]:
# Run experiments to find optimal regressor for each dataset
drug_results = find_best_regressor(
    X_train_drugs,
    y_train_drugs,
    num_folds = 4
)
opioid_results = find_best_regressor(
    X_train_opioids,
    y_train_opioids,
    num_folds = 4
)
stim_results = find_best_regressor(
    X_train_stims,
    y_train_stims,
    num_folds = 4
)

[I 2026-06-10 00:03:11,632] A new study created in memory with name: no-name-2ea7c0c1-f7cf-48c4-8d94-f2274c7a96cd
[I 2026-06-10 00:03:11,633] A new study created in memory with name: no-name-a541a452-22b5-4d96-8de0-e41064818d6a


[LightGBM] [Warning] min_data_in_leaf is set=6, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=6
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] min_data_in_leaf is set=6, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=6
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3770
[LightGBM] [Info] Number of data points in the train set: 2907, number of used features: 28
[LightGBM] [Info] Using GPU Device: NVIDIA L4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 25 dense feature groups (0.08 MB) transferred to GPU in 0.001493 secs. 1 sparse feature groups
[LightGBM] [Warning] min_data_in

[I 2026-06-10 00:03:30,197] Trial 0 finished with value: 2.9262579812617266 and parameters: {'num_leaves': 17, 'min_data_in_leaf': 6, 'learning_rate': 0.0035125206055828463, 'subsample': 0.9, 'colsample_bytree': 0.9, 'max_depth': 7, 'min_child_weight': 3, 'reg_alpha': 0.0005000674700083984, 'reg_lambda': 0.00014382244686350182}. Best is trial 0 with value: 2.9262579812617266.


Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

[I 2026-06-10 00:03:44,134] Trial 1 finished with value: 2.9755476520339066 and parameters: {'num_leaves': 108, 'min_data_in_leaf': 55, 'learning_rate': 0.0020466986013683527, 'subsample': 0.6, 'colsample_bytree': 0.8, 'max_depth': 7, 'min_child_weight': 3, 'reg_alpha': 0.06988395264828774, 'reg_lambda': 1.93054664922082}. Best is trial 0 with value: 2.9262579812617266.


...

[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 24 dense feature groups (0.03 MB) transferred to GPU in 0.001251 secs. 1 sparse feature groups
[LightGBM] [Warning] min_data_in_leaf is set=25, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=25
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] min_data_in_leaf is set=32, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=32
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] min_data_in_leaf is set=32, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=32
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 3770
[LightGBM] [Info] Number of data points in the train set: 2907, number

Exception ignored on calling ctypes callback function: <function _log_callback at 0x7c49ab778fe0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/basic.py", line 287, in _log_callback
    def _log_callback(msg: bytes) -> None:
    
KeyboardInterrupt: 


Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 25 dense feature groups (0.04 MB) transferred to GPU in 0.001247 secs. 1 sparse feature groups
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 25 dense feature groups (0.04 MB) transferred to GPU in 0.001276 secs. 1 sparse feature groups
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 25 dense feature groups (0.04 MB) transferred to GPU in 0.001234 secs. 1 sparse feature groups
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 25 dense feature groups (0.04 MB) transferred to GPU in 0.001286 secs. 1 sparse feature groups
[

Exception ignored on calling ctypes callback function: <function _log_callback at 0x7c49ab778fe0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/lightgbm/basic.py", line 287, in _log_callback
    def _log_callback(msg: bytes) -> None:
    
KeyboardInterrupt: 


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 25 dense feature groups (0.04 MB) transferred to GPU in 0.001692 secs. 1 sparse feature groups
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 25 dense feature groups (0.04 MB) transferred to GPU in 0.001631 secs. 1 sparse feature groups
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 25 dense feature groups (0.04 MB) transferred to GPU in 0.001683 secs. 1 sparse feature groups
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 25 dense feature groups (0.04 MB) transferred to GPU in 0.001547 secs. 1 sparse feature groups
[LightGBM] [Warning] No further splits with positive